# Add Hydrologic-Exposure Variables (PRISM Precipitation)

Downloads PRISM 4-km daily precipitation grids for the Helene and Milton landing windows, aggregates to county and cluster level, and merges into the pooled dataset for use as a hydrologic-exposure predictor.

**Variables produced (per county and per Helene cluster):**
- `precip_total_3day`  — total mm of rainfall in the 3-day window centered on landing
- `precip_total_7day`  — total mm in the landing-week window

**Windows:**
- Helene (landing 2024-09-26): 3-day = Sep 25–27, 7-day = Sep 25 – Oct 1
- Milton (landing 2024-10-09): 3-day = Oct 08–10, 7-day = Oct 08 – Oct 14

**Source:** PRISM Daily 4 km, free public download from `services.nacse.org/prism/data/get/us/4km/ppt/YYYYMMDD`. No API key required; rate-limit to ~1 request/sec to be polite.

**Why this notebook exists:** the global pooled OLS finds `dist_to_track_mi` has the *wrong* sign on `largest_drop_within` (β=−2.16, p=0.027) — farther counties drop more, opposite of wind-decay intuition. We hypothesize this is because wind-track distance fails to capture Helene's inland-flooding signature. If 7-day precipitation absorbs the puzzle (β on `dist_to_track_mi` → non-sig once `precip_total_7day` enters the model), we have evidence the misspecification is hydrologic.

**Bugfix 2026-05-18:** PRISM deprecated the old `data/public/4km/ppt/` endpoint; it now returns a 430-byte HTML deprecation notice that fails as a zip. Switched to the new endpoint `data/get/us/4km/ppt/` which returns a GeoTIFF zip (~1.5 MB). The download function now checks `Content-Type: application/zip` before saving.

In [1]:
# Install rasterstats if not present (run once)
import importlib
if importlib.util.find_spec("rasterstats") is None:
    import subprocess, sys
    print("Installing rasterstats...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rasterstats"])
    print("Done — restart kernel before continuing.")

In [2]:
import os
import time
import io
import zipfile
import warnings
import requests
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rasterstats
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

EXPOSURE_DIR = Path("../results/exposure/")
PRISM_RAW    = EXPOSURE_DIR / "prism_raw"
EXPOSURE_DIR.mkdir(parents=True, exist_ok=True)
PRISM_RAW.mkdir(parents=True, exist_ok=True)
print(f"Exposure dir: {EXPOSURE_DIR.resolve()}")

Exposure dir: /Users/qing/Library/CloudStorage/OneDrive-ColumbiaUniversityIrvingMedicalCenter/4_hurricane_category/results/exposure


## §1. Download PRISM daily ppt grids

PRISM caps download requests; we sleep 1 s between calls and cache the zips so re-runs are free. The download function rejects non-zip responses (HTML deprecation notices etc.) before writing them to disk.

In [3]:
# Helene 7-day window: Sep 25 – Oct 1, 2024
# Milton 7-day window: Oct 8  – Oct 14, 2024
helene_dates = pd.date_range("2024-09-25", "2024-10-01")
milton_dates = pd.date_range("2024-10-08", "2024-10-14")
all_dates = helene_dates.union(milton_dates)
print(f"Dates to download: {len(all_dates)} unique")
print(f"  Helene window: {helene_dates[0].date()} → {helene_dates[-1].date()}")
print(f"  Milton window: {milton_dates[0].date()} → {milton_dates[-1].date()}")

# NEW endpoint (post-2026 PRISM reorganization). The old `data/public/4km/ppt/` returns
# a 430-byte HTML deprecation page; this one returns the actual GeoTIFF zip (~1.5 MB).
PRISM_URL = "https://services.nacse.org/prism/data/get/us/4km/ppt/{ymd}"

def download_one(date, sleep_s=1.0, retries=3, min_bytes=10_000):
    ymd = date.strftime("%Y%m%d")
    zip_path = PRISM_RAW / f"prism_ppt_4km_{ymd}.zip"
    if zip_path.exists() and zip_path.stat().st_size > min_bytes:
        return zip_path, "cached"
    url = PRISM_URL.format(ymd=ymd)
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=60, headers={"User-Agent": "research-script"})
            r.raise_for_status()
            ctype = r.headers.get("Content-Type", "").lower()
            if "zip" not in ctype:
                # Server returned something else (HTML deprecation, error page, etc.)
                snippet = r.content[:200].decode("utf-8", errors="replace")
                return None, f"non-zip response (Content-Type={ctype!r}): {snippet!r}"
            if len(r.content) < min_bytes:
                return None, f"response too small ({len(r.content)} bytes) — likely an error page"
            zip_path.write_bytes(r.content)
            time.sleep(sleep_s)
            return zip_path, "downloaded"
        except Exception as e:
            if attempt == retries - 1:
                return None, f"failed: {e}"
            time.sleep(5)

results = []
for d in all_dates:
    path, status = download_one(d)
    results.append((d.strftime("%Y-%m-%d"), status, str(path) if path else None))
    print(f"  {d.date()}: {status}")
print(f"\nDownload summary: {sum(1 for _, s, _ in results if s in ('cached', 'downloaded'))} ok / {len(results)} total")

Dates to download: 14 unique
  Helene window: 2024-09-25 → 2024-10-01
  Milton window: 2024-10-08 → 2024-10-14
  2024-09-25: downloaded
  2024-09-26: non-zip response (Content-Type='application/octet-stream'): 'You have tried to download the file prism_ppt_us_25m_20240926.zip more than twice in one day (Pacific local time).  Note that repeated offenses may result in your IP address being blocked.'
  2024-09-27: downloaded
  2024-09-28: downloaded
  2024-09-29: downloaded
  2024-09-30: downloaded
  2024-10-01: downloaded
  2024-10-08: downloaded
  2024-10-09: downloaded
  2024-10-10: downloaded
  2024-10-11: downloaded
  2024-10-12: downloaded
  2024-10-13: downloaded
  2024-10-14: downloaded

Download summary: 13 ok / 14 total


In [4]:
# Extract zips → GeoTIFF files in PRISM_RAW (idempotent)
# New PRISM packaging contains .tif (GeoTIFF) instead of .bil. Files inside the zip
# are named like `prism_ppt_us_25m_YYYYMMDD.tif` (the '25m' is a PRISM internal id, not
# spatial resolution — the data really is 4 km).
for d in all_dates:
    ymd = d.strftime("%Y%m%d")
    zp = PRISM_RAW / f"prism_ppt_4km_{ymd}.zip"
    if not zp.exists():
        continue
    existing_tifs = list(PRISM_RAW.glob(f"prism_ppt_*_{ymd}.tif"))
    if existing_tifs:
        continue
    try:
        with zipfile.ZipFile(zp) as zf:
            zf.extractall(PRISM_RAW)
    except zipfile.BadZipFile as e:
        print(f"  ⚠ {zp.name}: BadZipFile — delete and re-download: {e}")
        continue

tif_files = sorted(PRISM_RAW.glob("prism_ppt_*.tif"))
print(f"Extracted {len(tif_files)} GeoTIFF rasters")
for f in tif_files[:3]:
    with rasterio.open(f) as ds:
        print(f"  {f.name}: shape={ds.shape}, CRS={ds.crs}, nodata={ds.nodata}, units={ds.units}")

Extracted 13 GeoTIFF rasters
  prism_ppt_us_25m_20240925.tif: shape=(621, 1405), CRS=EPSG:4269, nodata=-9999.0, units=(None,)
  prism_ppt_us_25m_20240927.tif: shape=(621, 1405), CRS=EPSG:4269, nodata=-9999.0, units=(None,)
  prism_ppt_us_25m_20240928.tif: shape=(621, 1405), CRS=EPSG:4269, nodata=-9999.0, units=(None,)


## §2. Define the study counties

Helene (271 counties) ∪ Milton (21 counties) = the universe we need rainfall for.

In [5]:
# Load study counties: Helene 271 + Milton 21
ca_hel = pd.read_csv("../results/helene_clustering/county_cluster_assignments.csv")
ca_hel["GEOID"] = ca_hel["GEOID"].astype(int)
mil_meta = pd.read_csv("../results/local_level/milton/county_metadata.csv")
mil_meta["GEOID"] = mil_meta["GEOID"].astype(int)

study_geoids_hel = set(ca_hel["GEOID"])
study_geoids_mil = set(mil_meta["GEOID"])
study_geoids = study_geoids_hel | study_geoids_mil
print(f"Helene: {len(study_geoids_hel)} counties")
print(f"Milton: {len(study_geoids_mil)} counties")
print(f"Union (some overlap possible): {len(study_geoids)} unique counties")
print(f"  Overlap Helene ∩ Milton: {len(study_geoids_hel & study_geoids_mil)}")

# Load county polygons; reproject to match PRISM CRS
county_shp = "./../../hurricane_oct/data/county_geo/tl_2023_us_county/tl_2023_us_county.shp"
counties_all = gpd.read_file(county_shp)
counties_all["GEOID"] = counties_all["GEOID"].astype(int)
counties = counties_all[counties_all["GEOID"].isin(study_geoids)].copy()

# Read PRISM CRS from one of the rasters and reproject counties to match
with rasterio.open(tif_files[0]) as ds:
    prism_crs = ds.crs
print(f"PRISM CRS: {prism_crs}")
print(f"Counties CRS before reprojection: {counties.crs}")
counties = counties.to_crs(prism_crs)
print(f"Counties CRS after reprojection: {counties.crs}")
print(f"N counties to process: {len(counties)}")

Helene: 271 counties
Milton: 21 counties
Union (some overlap possible): 292 unique counties
  Overlap Helene ∩ Milton: 0
PRISM CRS: EPSG:4269
Counties CRS before reprojection: EPSG:4269
Counties CRS after reprojection: GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101004,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4269"]]
N counties to process: 292


## §3. Zonal mean precipitation per county per day

In [6]:
# Filename pattern is `prism_ppt_us_25m_YYYYMMDD.tif` → ymd is the last underscore-token
records = []
for f in tif_files:
    ymd = f.stem.split("_")[-1]
    print(f"  {ymd}...", end=" ", flush=True)
    stats = rasterstats.zonal_stats(counties, str(f), stats="mean",
                                     nodata=-9999, all_touched=False)
    for geoid, s in zip(counties["GEOID"].values, stats):
        records.append({"GEOID": int(geoid), "date": ymd,
                         "ppt_mm": s["mean"] if s and s["mean"] is not None else np.nan})
    print("done")

ppt_daily = pd.DataFrame(records)
ppt_daily["date"] = pd.to_datetime(ppt_daily["date"])
out_path = EXPOSURE_DIR / "county_ppt_daily.csv"
ppt_daily.to_csv(out_path, index=False)
print(f"\nSaved daily county values: {out_path}")
print(f"Rows: {len(ppt_daily):,}  ({ppt_daily['GEOID'].nunique()} counties × {ppt_daily['date'].nunique()} days)")
print(f"\nSummary:")
print(ppt_daily["ppt_mm"].describe().round(2))

  20240925... done
  20240927... done
  20240928... done
  20240929... done
  20240930... done
  20241001... done
  20241008... done
  20241009... done
  20241010... done
  20241011... done
  20241012... done
  20241013... done
  20241014... done

Saved daily county values: ../results/exposure/county_ppt_daily.csv
Rows: 3,796  (292 counties × 13 days)

Summary:
count    3796.00
mean        9.78
std        27.90
min         0.00
25%         0.00
50%         0.00
75%         3.11
max       275.44
Name: ppt_mm, dtype: float64


## §4. 3-day and 7-day window totals per hurricane

In [7]:
def window_total(df, start, end):
    sub = df[(df["date"] >= start) & (df["date"] <= end)]
    return sub.groupby("GEOID")["ppt_mm"].sum()

rows = []
h3 = window_total(ppt_daily, "2024-09-25", "2024-09-27")
h7 = window_total(ppt_daily, "2024-09-25", "2024-10-01")
for geoid in h3.index:
    if geoid in study_geoids_hel:
        rows.append({"GEOID": geoid, "hurricane": "helene",
                     "precip_total_3day": h3.loc[geoid],
                     "precip_total_7day": h7.loc[geoid] if geoid in h7.index else np.nan})
m3 = window_total(ppt_daily, "2024-10-08", "2024-10-10")
m7 = window_total(ppt_daily, "2024-10-08", "2024-10-14")
for geoid in m3.index:
    if geoid in study_geoids_mil:
        rows.append({"GEOID": geoid, "hurricane": "milton",
                     "precip_total_3day": m3.loc[geoid],
                     "precip_total_7day": m7.loc[geoid] if geoid in m7.index else np.nan})

exposure_county = pd.DataFrame(rows)
out_path = EXPOSURE_DIR / "county_precip_exposure.csv"
exposure_county.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Rows: {len(exposure_county)} (Helene {(exposure_county['hurricane']=='helene').sum()} + Milton {(exposure_county['hurricane']=='milton').sum()})")
print("\nBy hurricane:")
print(exposure_county.groupby("hurricane")[["precip_total_3day", "precip_total_7day"]].describe().round(1))

Saved: ../results/exposure/county_precip_exposure.csv
Rows: 292 (Helene 271 + Milton 21)

By hurricane:
          precip_total_3day                                                \
                      count   mean   std   min   25%    50%    75%    max   
hurricane                                                                   
helene                271.0   80.7  48.4  15.2  41.2   70.4  119.6  242.7   
milton                 21.0  167.7  84.8  50.1  92.1  158.2  250.9  297.3   

          precip_total_7day                                                
                      count   mean   std   min   25%    50%    75%    max  
hurricane                                                                  
helene                271.0  118.5  37.0  33.6  91.9  118.6  135.8  305.5  
milton                 21.0  172.1  84.7  58.4  95.0  158.3  251.0  298.2  


## §5. Validation — spot-check against known truth

If these numbers are wildly off, debug the raster aggregation before going further.

In [8]:
TRUTH = [
    ("Buncombe NC (Asheville)",  37021, "helene", (350, 600)),
    ("Yancey NC",                37199, "helene", (400, 700)),
    ("Avery NC",                 37011, "helene", (350, 700)),
    ("Mitchell NC",              37121, "helene", (350, 650)),
    ("Taylor FL (Helene landfall)",  12123, "helene", (150, 350)),
    ("Dixie FL",                 12029, "helene", (150, 300)),
    ("Pinellas FL",              12103, "milton", (150, 350)),
    ("Hillsborough FL",          12057, "milton", (200, 400)),
    ("Sarasota FL",              12115, "milton", (150, 350)),
]

print(f"{'County':<32} {'expected':>14} {'observed_mm':>12} {'verdict'}")
print("-" * 80)
for name, geoid, h, (lo, hi) in TRUTH:
    sub = exposure_county[(exposure_county["GEOID"] == geoid) & (exposure_county["hurricane"] == h)]
    if len(sub) == 0:
        print(f"{name:<32} {f'[{lo}-{hi}]':>14} {'NOT IN SET':>12}")
        continue
    obs = sub.iloc[0]["precip_total_7day"]
    ok = "✓" if (lo * 0.7 <= obs <= hi * 1.3) else "✗"
    print(f"{name:<32} {f'[{lo}-{hi}]':>14} {obs:>12.1f} {ok}")

County                                 expected  observed_mm verdict
--------------------------------------------------------------------------------
Buncombe NC (Asheville)               [350-600]   NOT IN SET
Yancey NC                             [400-700]   NOT IN SET
Avery NC                              [350-700]   NOT IN SET
Mitchell NC                           [350-650]   NOT IN SET
Taylor FL (Helene landfall)           [150-350]         77.9 ✗
Dixie FL                              [150-300]         60.9 ✗
Pinellas FL                           [150-350]        297.3 ✓
Hillsborough FL                       [200-400]        275.1 ✓
Sarasota FL                           [150-350]        112.9 ✓


## §6. Aggregate to Helene clusters (population-weighted)

Population weighting matters here because cluster sizes vary from 1 to 29 counties.
A 29-county rural cluster's *mean* rainfall over-weights tiny rural counties; population weighting reflects where people actually experience the rainfall.

In [9]:
acs = pd.read_csv("acs_socioeconomic_v2.csv")
acs["GEOID"] = acs["GEOID"].astype(int)

hel_exposure = exposure_county[exposure_county["hurricane"] == "helene"].copy()
hel_exposure = hel_exposure.merge(ca_hel[["GEOID", "cluster"]], on="GEOID", how="left")
hel_exposure = hel_exposure.merge(acs[["GEOID", "total_population"]], on="GEOID", how="left")

missing = hel_exposure[hel_exposure["total_population"].isna()]
if len(missing):
    print(f"Warning: {len(missing)} counties missing population — using equal weights for those")
    hel_exposure["total_population"] = hel_exposure["total_population"].fillna(1)

def pop_weighted(group, col):
    w = group["total_population"].values
    v = group[col].values
    if w.sum() == 0:
        return v.mean()
    return np.sum(w * v) / w.sum()

cluster_exposure_hel = hel_exposure.groupby("cluster").apply(
    lambda g: pd.Series({
        "precip_total_3day": pop_weighted(g, "precip_total_3day"),
        "precip_total_7day": pop_weighted(g, "precip_total_7day"),
        "n_counties": len(g),
        "total_pop": g["total_population"].sum(),
    })
).reset_index()
cluster_exposure_hel["hurricane"] = "helene"

out_path = EXPOSURE_DIR / "cluster_precip_exposure_helene.csv"
cluster_exposure_hel.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"\nHelene cluster precip stats:")
print(cluster_exposure_hel[["precip_total_3day", "precip_total_7day"]].describe().round(1))

Saved: ../results/exposure/cluster_precip_exposure_helene.csv

Helene cluster precip stats:
       precip_total_3day  precip_total_7day
count               38.0               38.0
mean                75.3              113.4
std                 45.8               34.6
min                 16.2               33.6
25%                 34.6               96.4
50%                 64.5              117.5
75%                109.2              127.0
max                213.8              222.9


## §7. Merge into pooled dataset

Adds the two precip variables to `pooled_dataset.csv`. Writes a new file (`pooled_dataset_with_precip.csv`) without overwriting the original.

In [10]:
pooled = pd.read_csv("../results/local_level/regression/pooled_dataset.csv")
print(f"Original pooled: N={len(pooled)} ({pooled['hurricane'].value_counts().to_dict()})")

# Milton rows: NAME = county name → merge via county metadata to get GEOID
mil_meta_full = pd.read_csv("../results/local_level/milton/county_metadata.csv")
mil_meta_full["GEOID"] = mil_meta_full["GEOID"].astype(int)
mil_with_geoid = pooled[pooled["hurricane"] == "milton"].merge(
    mil_meta_full[["GEOID", "NAME"]], on="NAME", how="left")

mil_precip = exposure_county[exposure_county["hurricane"] == "milton"][
    ["GEOID", "precip_total_3day", "precip_total_7day"]]
mil_with_precip = mil_with_geoid.merge(mil_precip, on="GEOID", how="left")

hel_rows = pooled[pooled["hurricane"] == "helene"].copy()
hel_rows["cluster"] = hel_rows["NAME"].str.replace("Cluster_", "").astype(int)
hel_with_precip = hel_rows.merge(
    cluster_exposure_hel[["cluster", "precip_total_3day", "precip_total_7day"]],
    on="cluster", how="left").drop(columns=["cluster"])

mil_with_precip = mil_with_precip.drop(columns=["GEOID"], errors="ignore")

pooled_aug = pd.concat([mil_with_precip, hel_with_precip], ignore_index=True)
out_path = "../results/local_level/regression/pooled_dataset_with_precip.csv"
pooled_aug.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Augmented dataset: N={len(pooled_aug)}, columns added: precip_total_3day, precip_total_7day")
print(f"Missing precip values: {pooled_aug[['precip_total_3day','precip_total_7day']].isna().sum().to_dict()}")
print("\nSample rows (sorted by 7-day precip):")
display_cols = ["NAME", "hurricane", "dist_to_track_mi",
                 "precip_total_3day", "precip_total_7day", "largest_drop_within"]
print(pooled_aug.sort_values("precip_total_7day", ascending=False)[display_cols].head(10).to_string(index=False))

Original pooled: N=59 ({'helene': 38, 'milton': 21})
Saved: ../results/local_level/regression/pooled_dataset_with_precip.csv
Augmented dataset: N=59, columns added: precip_total_3day, precip_total_7day
Missing precip values: {'precip_total_3day': 0, 'precip_total_7day': 0}

Sample rows (sorted by 7-day precip):
        NAME hurricane  dist_to_track_mi  precip_total_3day  precip_total_7day  largest_drop_within
     Volusia    milton         48.803460         281.551420         298.179206           -22.196909
    Pinellas    milton         41.477201         297.290115         297.290188           -32.934764
Hillsborough    milton         24.427360         275.028981         275.052900           -42.276486
       Pasco    milton         51.809095         264.537123         264.545912           -40.525795
        Lake    milton         46.011896         250.998153         254.135468           -20.407382
      Sumter    milton         56.823550         250.926060         250.953766         

## §8. Does precip absorb the distance-to-track puzzle?

The headline question. Refit the pooled OLS first without and then with `precip_total_7day`. Watch the coefficient on `dist_to_track_mi`.

**Verdict scenarios:**
1. `dist_to_track_mi` becomes non-significant after adding precip + precip is significant negative → **clean win**, hydrologic story confirmed
2. Both stay significant → precip captures additional variance but track distance is still measuring something else (wind? topography?)
3. Precip is non-significant → surprising; would point to a different exposure mechanism

In [11]:
BASE_FEATURES = [
    "median_household_income", "pct_no_vehicle", "pct_white", "nchs_code",
    "total_population", "pop_density", "dist_to_track_mi",
    "insurance_coverage_pct", "is_milton", "is_coastal",
]
DV = "largest_drop_within"

def fit_ols(df, features, dv):
    d = df.dropna(subset=features + [dv]).reset_index(drop=True)
    Xz = pd.DataFrame(StandardScaler().fit_transform(d[features]),
                       columns=features, index=d.index)
    X = sm.add_constant(Xz)
    return sm.OLS(d[dv], X).fit(), len(d)

mA, nA = fit_ols(pooled_aug, BASE_FEATURES, DV)
mB, nB = fit_ols(pooled_aug, BASE_FEATURES + ["precip_total_7day"], DV)
mC, nC = fit_ols(pooled_aug, BASE_FEATURES + ["precip_total_3day", "precip_total_7day"], DV)

for label, m, n in [("A: baseline (no precip)", mA, nA),
                      ("B: + precip_total_7day", mB, nB),
                      ("C: + 3day + 7day",        mC, nC)]:
    print(f"\n{'=' * 78}")
    print(f"Model {label}  (N={n}, R²={m.rsquared:.3f}, Adj.R²={m.rsquared_adj:.3f}, F p={m.f_pvalue:.4e})")
    print("=" * 78)
    print(f"  {'Variable':<28} {'β':>9} {'p':>9}")
    print("  " + "-" * 48)
    for nm, b, p in zip(m.params.index, m.params.values, m.pvalues.values):
        sig = "**" if p < 0.05 else "*" if p < 0.10 else ""
        print(f"  {nm:<28} {b:>9.3f} {p:>9.4f} {sig}")

print("\n" + "#" * 78)
print("# HEADLINE: dist_to_track_mi coefficient across models")
print("#" * 78)
for label, m in [("A baseline", mA), ("B + 7day", mB), ("C + 3day+7day", mC)]:
    if "dist_to_track_mi" in m.params.index:
        b = m.params["dist_to_track_mi"]; p = m.pvalues["dist_to_track_mi"]
        sig = "**" if p < 0.05 else "*" if p < 0.10 else "NS"
        print(f"  {label:<18} β={b:+.3f}  p={p:.4f}  [{sig}]")
if "precip_total_7day" in mB.params.index:
    b = mB.params["precip_total_7day"]; p = mB.pvalues["precip_total_7day"]
    sig = "**" if p < 0.05 else "*" if p < 0.10 else "NS"
    print(f"  precip_7day in B  β={b:+.3f}  p={p:.4f}  [{sig}]")


Model A: baseline (no precip)  (N=59, R²=0.639, Adj.R²=0.563, F p=9.2078e-08)
  Variable                             β         p
  ------------------------------------------------
  const                          -21.662    0.0000 **
  median_household_income         -0.370    0.7876 
  pct_no_vehicle                  -0.525    0.6840 
  pct_white                        0.051    0.9685 
  nchs_code                       -0.847    0.7163 
  total_population                 0.601    0.6641 
  pop_density                     -0.998    0.5035 
  dist_to_track_mi                -2.158    0.0267 **
  insurance_coverage_pct           1.375    0.1938 
  is_milton                       -7.353    0.0000 **
  is_coastal                      -1.191    0.2959 

Model B: + precip_total_7day  (N=59, R²=0.643, Adj.R²=0.559, F p=2.2259e-07)
  Variable                             β         p
  ------------------------------------------------
  const                          -21.662    0.0000 **
  media

## §9. Next steps

Depending on §8 verdict:

- **Verdict 1 (clean win):** Update [findings.md](../notes/findings.md) §"Distance-to-track puzzle" with the absorption result; add `precip_total_7day` to the `FEATURES` list in `helene_gwr.ipynb` and re-run GWR to see whether the coastal-vs-inland β localization collapses now that the misspecification is fixed.
- **Verdict 2 (partial):** Keep both variables in models; discuss residual track-distance effect as proxy for non-precipitation exposure (wind field, topography). Run GWR with precip included as a covariate.
- **Verdict 3 (precip not sig):** Sanity-check the PRISM aggregation; consider an alternative exposure variable (Sentinel-1 SAR flood extent, FEMA disaster declarations).

Independent of verdict, also update `findings.md` §"Sensitivity to Okeechobee" with the new column counts and the precip-inclusion sensitivity.